In [1]:
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.preprocessing import LabelEncoder

In [2]:
# ---- Load clean data ----
df = pd.read_csv("../data/processed/clean_matches.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

print(f"Loaded {len(df)} clean matches")
print(df[["date", "home_team", "away_team", "result"]].head())

Loaded 32314 clean matches
        date home_team  away_team    result
0 1990-01-12   Algeria       Mali  Home Win
1 1990-01-14   Algeria   Cameroon  Home Win
2 1990-01-17    Greece    Belgium  Home Win
3 1990-01-17    Mexico  Argentina  Home Win
4 1990-01-20    Malawi   Tanzania      Draw


In [3]:
# ---- Encode team names to numbers ----
le = LabelEncoder()
all_teams = pd.concat([df["home_team"], df["away_team"]]).unique()
le.fit(all_teams)

df["home_team_enc"] = le.transform(df["home_team"])
df["away_team_enc"] = le.transform(df["away_team"])

print(f"\nEncoded {len(le.classes_)} unique teams")
print(df[["home_team", "home_team_enc"]].drop_duplicates().head(5).to_string())



Encoded 326 unique teams
  home_team  home_team_enc
0   Algeria              4
2    Greece            114
3    Mexico            183
4    Malawi            169
5  Eswatini             93


In [4]:
# ---- Calculate rolling team form (win rate, last N matches) ----

N = 10  # number of past matches to consider

team_history = {team: [] for team in all_teams}  # team -> list of 0/1 (loss/win) with dates
form_cache = {}  # (team, row_index) -> win rate, filled as we go

home_form = np.zeros(len(df))
away_form = np.zeros(len(df))

for i, row in df.iterrows():
    home, away, result = row["home_team"], row["away_team"], row["result"]

    home_hist = team_history[home][-N:]
    away_hist = team_history[away][-N:]

    home_form[i] = np.mean(home_hist) if home_hist else 0.5
    away_form[i] = np.mean(away_hist) if away_hist else 0.5

    if result == "Home Win":
        team_history[home].append(1)
        team_history[away].append(0)
    elif result == "Away Win":
        team_history[home].append(0)
        team_history[away].append(1)
    else:
        team_history[home].append(0.5)
        team_history[away].append(0.5)

    if i % 5000 == 0:
        print(f"Processed {i}/{len(df)} rows")

df["home_form"] = home_form.round(4)
df["away_form"] = away_form.round(4)

print("\nForm calculation complete")
print(df[["home_team", "away_team", "home_form", "away_form"]].head(10))

Processed 0/32314 rows
Processed 5000/32314 rows
Processed 10000/32314 rows
Processed 15000/32314 rows
Processed 20000/32314 rows
Processed 25000/32314 rows
Processed 30000/32314 rows

Form calculation complete
  home_team  away_team  home_form  away_form
0   Algeria       Mali       0.50       0.50
1   Algeria   Cameroon       1.00       0.50
2    Greece    Belgium       0.50       0.50
3    Mexico  Argentina       0.50       0.50
4    Malawi   Tanzania       0.50       0.50
5  Eswatini   Botswana       0.50       0.50
6  Botswana     Malawi       0.00       0.50
7    Kuwait     France       0.50       0.50
8  Eswatini   Tanzania       1.00       0.50
9  Botswana   Tanzania       0.25       0.75


In [ ]:
# ---- Calculate head-to-head win rate ----
h2h_history = {}  # (teamA, teamB) sorted tuple -> list of results from teamA's perspective
home_h2h = np.zeros(len(df))

for i, row in df.iterrows():
    home, away, result = row["home_team"], row["away_team"], row["result"]
    key = tuple(sorted([home, away]))

    past = h2h_history.get(key, [])
    if past:
        # past entries are stored as (winner_name) or "Draw"
        home_wins = sum(1 for w in past if w == home)
        home_h2h[i] = home_wins / len(past)
    else:
        home_h2h[i] = 0.5

    if result == "Home Win":
        winner = home
    elif result == "Away Win":
        winner = away
    else:
        winner = "Draw"

    h2h_history.setdefault(key, []).append(winner)

df["home_h2h_rate"] = home_h2h.round(4)

print("\nHead-to-head feature added")
print(df[["home_team", "away_team", "home_h2h_rate"]].head(10))


Head-to-head feature added
  home_team  away_team  home_h2h_rate
0   Algeria       Mali            0.5
1   Algeria   Cameroon            0.5
2    Greece    Belgium            0.5
3    Mexico  Argentina            0.5
4    Malawi   Tanzania            0.5
5  Eswatini   Botswana            0.5
6  Botswana     Malawi            0.5
7    Kuwait     France            0.5
8  Eswatini   Tanzania            0.5
9  Botswana   Tanzania            0.5


In [6]:
# ---- Add remaining features ----
df["is_neutral"] = df["neutral"].astype(int)
df["form_diff"] = df["home_form"] - df["away_form"]

In [7]:
# ---- Final feature set + integrity checks ----
feature_cols = [
    "home_team_enc", "away_team_enc",
    "home_form", "away_form", "form_diff",
    "home_h2h_rate", "is_neutral"
]

print("\n=== Final feature columns ===")
print(df[feature_cols + ["result"]].head())

null_check = df[feature_cols + ["result"]].isnull().sum()
assert null_check.sum() == 0, f"Nulls found:\n{null_check[null_check > 0]}"
print("\nNo nulls in feature columns — integrity check passed")

print("\n=== Feature ranges (sanity check) ===")
print(df[feature_cols].describe().round(3))


=== Final feature columns ===
   home_team_enc  away_team_enc  home_form  away_form  form_diff  \
0              4            172        0.5        0.5        0.0   
1              4             46        1.0        0.5        0.5   
2            114             28        0.5        0.5        0.0   
3            183             13        0.5        0.5        0.0   
4            169            285        0.5        0.5        0.0   

   home_h2h_rate  is_neutral    result  
0            0.5           1  Home Win  
1            0.5           1  Home Win  
2            0.5           0  Home Win  
3            0.5           1  Home Win  
4            0.5           1      Draw  

No nulls in feature columns — integrity check passed

=== Feature ranges (sanity check) ===
       home_team_enc  away_team_enc  home_form  away_form  form_diff  \
count      32314.000      32314.000  32314.000  32314.000  32314.000   
mean         161.546        162.843      0.506      0.493      0.013   
std  

In [8]:
# ---- Save Outputs ----

os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../models", exist_ok=True)

df.to_csv("../data/processed/features.csv", index=False)
pickle.dump(le, open("../models/team_encoder.pkl", "wb"))

print(f"\nSaved {len(df)} rows with features to data/processed/features.csv")
print("Saved team encoder to models/team_encoder.pkl")
print(f"Feature columns: {feature_cols}")


Saved 32314 rows with features to data/processed/features.csv
Saved team encoder to models/team_encoder.pkl
Feature columns: ['home_team_enc', 'away_team_enc', 'home_form', 'away_form', 'form_diff', 'home_h2h_rate', 'is_neutral']
